# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access basic metadata (see documentation: https://mlcommons.github.io/croissant/api-docs/)
meta = dataset.metadata

print(f"Dataset Name: {meta.name}")
print(f"Dataset Version: {getattr(meta, 'version', 'N/A')}")
print(f"Description: {getattr(meta, 'description', 'No description provided.')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and fields
record_sets = dataset.metadata.record_sets
print("Available record sets (@id):\n-------------------------------")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', 'N/A')}")
    print()

# For demonstration, pick the first available record set:
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"Example record set for extraction: {example_record_set_id}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Compose a list of all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Extract all available data into pandas DataFrames using mlcroissant
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not extract records for {record_set_id}: {e}")

# Choose the first record set for exploration if available
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id]
    print(f"Columns in selected record set ({selected_record_set_id}):\n{df.columns.tolist()}")
    display(df.head())
else:
    selected_record_set_id = None
    print("No record sets available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA: Filtering, normalization, and grouping (with safety for missing fields)
import numpy as np

if selected_record_set_id is not None and not df.empty:
    # Try to auto-detect a likely numeric field by data type
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field for filtering: {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Group by a possible categorical field
        potential_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        potential_group_fields = [col for col in potential_group_fields if col != numeric_field]
        if potential_group_fields:
            group_field = potential_group_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field, dropna=True)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields available in this record set.")
else:
    print("Cannot perform EDA: No data available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if EDA identified a numeric field and filtered_df
if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of filtered '{numeric_field}' values")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouping was performed, show a barplot
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(12, 5))
        g = filtered_df.groupby(group_field)[numeric_field].mean()
        g = g.sort_values(ascending=False).head(15)  # top 15 for clarity
        sns.barplot(y=g.index, x=g.values, orient='h')
        plt.xlabel(f"Avg {numeric_field}")
        plt.ylabel(group_field)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains comprehensive information on protein abundance, modifications, and sequences originating from human extracellular vesicles isolated from stimulated mast cells.
- Record sets are accessible via their `@id`, and their fields and columns can be dynamically listed and loaded with `mlcroissant`.
- Numeric fields (such as peptide counts or normalized abundances) can be filtered and normalized for exploratory statistics and visualization.
- Grouping and plotting by categorical variables (when present) helps summarize data and reveal differences among groups.

For further scientific use, referencing all fields and record sets by their `@id` (as per Croissant best practices) ensures reproducibility and precision across research projects.